# 05 朴素贝叶斯 Naive Bayes

依赖安装说明：`pip install numpy matplotlib scikit-learn`

朴素贝叶斯常用于文本分类、垃圾邮件识别和简单高维分类。它的核心假设很强：给定类别后，各特征条件独立。


## 1. 数学逻辑

贝叶斯公式：

$$P(y|x)=\frac{P(x|y)P(y)}{P(x)}$$

分类时 `P(x)` 对所有类别一样，可以忽略：

$$\hat y = \arg\max_y P(y)P(x|y)$$

朴素假设把联合概率拆开：

$$P(x|y)=\prod_j P(x_j|y)$$

为了避免很多小概率相乘下溢，实际计算通常取 log：

$$\log P(y|x) \propto \log P(y) + \sum_j \log P(x_j|y)$$


In [ ]:
import numpy as np
from collections import Counter, defaultdict
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split

texts = [
    'win money now', 'cheap prize win', 'limited offer money', 'claim your prize',
    'meeting schedule today', 'project update today', 'please review document', 'team lunch schedule',
    'win cash prize', 'cheap money offer', 'document review meeting', 'project schedule update'
]
labels = np.array([1,1,1,1,0,0,0,0,1,1,0,0])  # 1=spam, 0=normal


In [ ]:
# 从零实现：Multinomial Naive Bayes 的核心计算
vocab = sorted(set(word for text in texts for word in text.split()))
word_to_id = {w: i for i, w in enumerate(vocab)}

alpha = 1.0  # Laplace smoothing，避免未见过的词概率为 0
classes = sorted(set(labels))
class_log_prior = {}
word_log_prob = {}

for c in classes:
    docs_c = [texts[i] for i in range(len(texts)) if labels[i] == c]
    class_log_prior[c] = np.log(len(docs_c) / len(texts))
    counts = np.zeros(len(vocab))
    for text in docs_c:
        for word in text.split():
            counts[word_to_id[word]] += 1
    probs = (counts + alpha) / (counts.sum() + alpha * len(vocab))
    word_log_prob[c] = np.log(probs)

def predict_text(text):
    scores = {}
    for c in classes:
        score = class_log_prior[c]
        for word in text.split():
            if word in word_to_id:
                score += word_log_prob[c][word_to_id[word]]
        scores[c] = score
    return max(scores, key=scores.get), scores

for text in ['win money prize', 'project meeting review']:
    pred, scores = predict_text(text)
    print(text, '->', 'spam' if pred == 1 else 'normal', scores)


In [ ]:
# sklearn 实战：CountVectorizer + MultinomialNB 是文本分类常见组合
X_train, X_test, y_train, y_test = train_test_split(texts, labels, random_state=42, stratify=labels)
vectorizer = CountVectorizer()
X_train_counts = vectorizer.fit_transform(X_train)
X_test_counts = vectorizer.transform(X_test)

model = MultinomialNB(alpha=1.0)
model.fit(X_train_counts, y_train)
pred = model.predict(X_test_counts)

print('词表:', vectorizer.get_feature_names_out())
print(classification_report(y_test, pred, target_names=['normal', 'spam']))


## 2. 常见误区

- “朴素”指条件独立假设很强，不代表模型没用；文本高维稀疏场景下它常常很强。
- 没有平滑时，测试文本出现训练中未见组合，概率可能被压成 0。
- 词袋模型忽略语序，所以它不能理解复杂句法。

## 3. 小实验

- 改 `alpha`，观察平滑强弱。
- 加入更多容易混淆的文本。
- 把词频换成 TF-IDF，再比较效果。
